# Shortest paths: linear optimization, Dijkstra and profiling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/aabw/notebooks/lecture-2/shortest-path-optimization-vs-algorithms.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/aabw/notebooks/lecture-2/shortest-path-optimization-vs-algorithms.ipynb)

AABW Lecture 2 · Joaquim Gromicho

Compare a linear-programming model with two Dijkstra implementations and
NetworkX on the same weighted problem. Then use a line profiler to investigate
where the implementations spend time. Allow about 30–45 minutes to read and
experiment; the optional large profiling run can take substantially longer.


## One problem, different computational approaches

Joaquim Gromicho, 2026

The shortest path problem is a classical example that sits at the intersection of Linear Optimization and Computer Science.

From the perspective of Linear Optimization, shortest path problems can be formulated naturally as linear programs. A standard formulation uses flow conservation constraints together with arc variables indicating whether an edge is used. Although the variables are naturally integer, the constraint matrix of the problem is totally unimodular. As a consequence, an integral optimal extreme point exists whenever the problem has an optimum. With ties, other optimal solutions can be fractional. In other words, the problem is inherently discrete but does not require explicitly forcing the variables to be integer.

This illustrates an important strength of linear optimization. Many graph problems can be expressed as linear models in a very natural way. Linear optimization therefore provides a unifying modelling framework capable of representing a wide range of combinatorial problems, including network flows, assignment, matching, and shortest paths.

However, while the linear programming formulation is elegant and conceptually powerful, in practice shortest path problems are almost always solved using specialized algorithms. Dedicated algorithms exploit the structure of graphs much more directly and are typically far more efficient than solving a general purpose linear program.

The most famous of these algorithms is Dijkstra's algorithm, introduced by Edsger W. Dijkstra in 1959. Dijkstra was a computer scientist, and his work exemplifies the algorithmic tradition within Computer Science that focuses on designing efficient procedures for specific problem classes.

It is also important to note that an algorithm is not just an abstract idea, its performance depends strongly on the way it is implemented and on the data structures that support it. For example, different priority queue implementations lead to different theoretical and practical running times. Binary heaps, Fibonacci heaps, and other structures change the complexity bounds and the empirical performance of Dijkstra's algorithm.

In summary, the shortest path problem highlights a productive interaction between Linear Optimization and Computer Science. Linear optimization provides a clean mathematical formulation and theoretical insight, while algorithmic approaches provide highly efficient computational methods tailored to the structure of the problem.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages


required_packages = {'pyomo': 'pyomo', 'highspy': 'highspy', 'networkx': 'networkx',
                     'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib',
                     'psutil': 'psutil'}
ensure_packages(required_packages)
from teaching_utils import solve_checked
import pyomo.environ as pyo
solver_name = 'appsi_highs'


The notebook uses the open-source HiGHS solver installed in the setup cell above; no separate solver download is needed.


In [ ]:
import pyomo.environ as pyo
import networkx as nx
import pandas as pd
import numpy as np
from io import StringIO
import matplotlib.pyplot as plt
from time import perf_counter as pc

In [ ]:
nodesFromLecture = '''
node;x;y
A;0;1
B;2;2
C;5;2
D;2;0
E;5;0
'''

edgesFromLecture = '''
from;to;length
A;B;10
A;D; 5
B;C; 1
B;D; 2
C;E; 4
D;B; 3
D;C; 9
D;E; 2
E;A; 7
E;C; 6
'''

edges = pd.read_csv(StringIO(edgesFromLecture), sep=";",index_col=['from','to'])
nodes = pd.read_csv(StringIO(nodesFromLecture), sep=";",index_col='node')

In [ ]:
nodes

In [ ]:
edges

In [ ]:
def FillGraphFromFrames( g, nodes, edges ):
    """ Fills a graph with nodes and edges and the corresponding attributes from data frames.
    The nodes frame is expected to be indexed on the node names and the edges frame on the tuples of nodes defining  the edges.
    Args:
        g (graph): either a nx.Graph or a nx.DiGraph object
        nodes (dataframe): one node per row, the columns define attribute values
        edges (dataframe): one edge per row, the columns define attribute values

    Returns:
        graph: the graph g taken as input extended with the nodes and edges from the dataframes
    """
    for node, row in nodes.iterrows():
        g.add_node(node,**row.to_dict())
    for edge, row in edges.iterrows():
        g.add_edge(*edge,**row.to_dict())
    return g

In [ ]:
def draw_graph(G: nx.Graph, solution : list = [ ]) -> None:
    """
    Draws a networkx graph using the specified node positions, edge lengths, and edge colors.

    :param G: A networkx Graph object with nodes that have 'x' and 'y' attributes for position
              and edges that have 'length' and 'color' attributes.
    """

    # Extract node positions from the node attributes
    pos = {node: (data['x'], data['y']) for node, data in G.nodes(data=True)}

    # Draw the graph using the node positions
    nx.draw_networkx_nodes(G, pos, node_size=500)
    nx.draw_networkx_labels(G, pos, font_color='lightblue', font_size=13)

    # Draw the graph edges with the specified colors and add edge labels with the length
    edge_colors = ['red' if (s,t) in solution else 'black' for s,t in G.edges]
    edge_labels = nx.get_edge_attributes(G, 'length')
    nx.draw_networkx_edges(G, pos, edge_color=edge_colors)
    nx.draw_networkx_edge_labels(G, pos, label_pos=.2, edge_labels=edge_labels)

    # Show the plot
    plt.show()

In [ ]:
G = FillGraphFromFrames( nx.DiGraph(), nodes, edges )

In [ ]:
draw_graph(G)

In [ ]:
G.nodes(data=True)

In [ ]:
G.edges(data=True)

## Send one unit of flow from the start to the target

For each directed arc $(i,j)$ with nonnegative length $c_{ij}$, let $x_{ij}$
be the amount of flow. For an undirected edge, include both directions.

$$\min_{x\geq0}\sum_{(i,j)\in E}c_{ij}x_{ij}$$

$$\sum_{j:(v,j)\in E}x_{vj}-\sum_{i:(i,v)\in E}x_{iv}
=\begin{cases}1&v=s,\\-1&v=t,\\0&\text{otherwise.}\end{cases}$$

One unit leaves the start, one arrives at the target, and other nodes conserve
flow. We use nonnegative continuous variables: a shortest-path optimum can be
obtained without declaring them binary. Equal-cost alternatives can give more
than one optimal route. Compare total lengths, not an arbitrary tie-breaking rule.

The examples use simple graphs with explicit, finite, nonnegative `length`
attributes. Missing lengths are input errors; they must not silently become zero
in the model or one in another algorithm.


In [ ]:
def validate_path_input(G, s, t, attribute='length'):
    if G.is_multigraph():
        raise ValueError('This lesson uses simple graphs, not parallel edges.')
    if s not in G or t not in G:
        raise nx.NodeNotFound('The start and target must be nodes of the graph.')
    for u, v, data in G.edges(data=True):
        if attribute not in data or not np.isfinite(data[attribute]) or data[attribute] < 0:
            raise ValueError(f'Edge {(u, v)} needs a finite, nonnegative {attribute}.')


In [ ]:
def ShortestPathAsLinearOptimization(G, s, t, attribute='length'):
    validate_path_input(G, s, t, attribute)
    successors = G.adj
    predecessors = G.pred if G.is_directed() else G.adj
    arcs = list(G.edges())
    if not G.is_directed():
        arcs += [(j, i) for i, j in G.edges() if i != j]
    m = pyo.ConcreteModel('Shortest path as one unit of flow')
    m.N = pyo.Set(initialize=list(G.nodes()))
    m.E = pyo.Set(initialize=arcs, dimen=2)
    m.x = pyo.Var(m.E, within=pyo.NonNegativeReals)
    m.c = pyo.Param(m.E, initialize=lambda m, i, j: G.edges[i, j][attribute])
    m.b = pyo.Param(m.N, initialize=lambda m, j: int(j == s) - int(j == t))
    m.length = pyo.Objective(expr=pyo.quicksum(m.c[e] * m.x[e] for e in m.E))

    @m.Constraint(m.N)
    def balance(m, j):
        return (pyo.quicksum(m.x[j, k] for k in successors[j])
                - pyo.quicksum(m.x[i, j] for i in predecessors[j])) == m.b[j]

    return m


In [ ]:
m = ShortestPathAsLinearOptimization(G, 'A', 'C')
results = solve_checked(m, solver_name, options={'threads': 1})
m.pprint()


In [ ]:
def SolveAsLO(G, s, t, solver=solver_name):
    m = ShortestPathAsLinearOptimization(G, s, t)
    solve_checked(m, solver, options={'threads': 1})
    return [edge for edge in m.E if pyo.value(m.x[edge]) > 0.5], pyo.value(m.length)


In [ ]:
sol, length = SolveAsLO( G, 'A', 'C' )
sol, length

In [ ]:
draw_graph( G, sol )

## Dijkstra with a simple minimum scan

Maintain tentative distances and a predecessor for each improved route. At each
step, make the smallest tentative label permanent. With nonnegative lengths,
that label cannot later improve. `min(...)` scans the candidate labels; repeating
this scan can take quadratic time in the number of nodes.

The linear model above returns used arcs for drawing. The algorithms below
return an ordered list of nodes. Both also return the total weighted length.


In [ ]:
def SimpleDijkstra( G, start, terminus, attribute='length' ):

    validate_path_input(G, start, terminus, attribute)

    def DijkstraInitialize(start):
        labels = dict()
        labels[start] = 0
        return start,{start},labels,dict()

    def DijkstraScan(G,current,labels,prev,attribute):
        for _,suc,length in G.edges(current,data=attribute):
            if labels[current]+length < labels.get(suc,np.inf):
                labels[suc] = labels[current]+length
                prev[suc] = current
        return labels,prev

    def DijkstraAdvance(labels,permanent):
        candidates = labels.keys() - permanent
        if not candidates:
            raise nx.NetworkXNoPath(f'No path from {start} to {terminus}.')
        return min(candidates, key=labels.get)

    def DijkstraBacktrack(current,prev,start):
        path = [current]
        while current != start:
            current = prev[current]
            path.append(current)
        path.reverse()
        return path

    current,permanent,labels,prev = DijkstraInitialize(start)
    while current != terminus:
        labels,prev = DijkstraScan(G,current,labels,prev,attribute)
        current = DijkstraAdvance(labels,permanent)
        permanent.add(current)
    return DijkstraBacktrack(current,prev,start),labels[terminus]


In [ ]:
SimpleDijkstra( G, 'A', 'C' )

## Dijkstra with a heap

A heap avoids scanning every candidate label to find the next minimum. Improved
labels are pushed onto the heap; obsolete entries are ignored when popped.
A counter breaks ties without requiring node labels to be comparable.
Store predecessors when a distance improves, rather than reconstructing a path
by testing floating-point distances for exact equality.


In [ ]:
def Dijkstra(G, start, terminus, attribute='length'):
    from heapq import heappush, heappop
    from itertools import count

    validate_path_input(G, start, terminus, attribute)
    order = count()
    pending = [(0, next(order), start)]
    distances = {start: 0}
    previous = {}
    permanent = set()
    while pending:
        distance, _, current = heappop(pending)
        if current in permanent:
            continue
        permanent.add(current)
        if current == terminus:
            path = [current]
            while current != start:
                current = previous[current]
                path.append(current)
            return path[::-1], distance
        for successor, data in G.adj[current].items():
            candidate = distance + data[attribute]
            if successor not in permanent and candidate < distances.get(successor, np.inf):
                distances[successor] = candidate
                previous[successor] = current
                heappush(pending, (candidate, next(order), successor))
    raise nx.NetworkXNoPath(f'No path from {start} to {terminus}.')


In [ ]:
Dijkstra( G, 'A', 'C' )

## Compare correct answers before comparing timings

Both graph topology and edge lengths use a seed. Every strategy receives the
same graph for a given size. We first check a warm-up run, then record three
wall-clock measurements and report their median. Every returned length must
match NetworkX's weighted reference value before a timing is accepted.

The clock includes each strategy's input validation and computation. For the
linear-programming strategy this includes building a fresh model and solver,
solving with one HiGHS thread and extracting the answer. It excludes graph
generation, the reference calculation, correctness checks and plotting. There
are no reused models or warm starts; warming up library code is different from
reusing a solved optimization model.

These are illustrative measurements in your current runtime. Shared-machine
load is uncontrolled, and medians of three repeats are not a definitive ranking.
Keep results from different machines or environments separate. Raw samples are
available in each timing table's `attrs['samples']` for further inspection.


In [ ]:
import os
import platform
from datetime import datetime, timezone
from importlib.metadata import version
import psutil
import highspy

environment = {
    'UTC': datetime.now(timezone.utc).isoformat(),
    'Python': platform.python_version(),
    'OS': platform.platform(),
    'Architecture': platform.machine(),
    'CPU model': platform.processor() or 'not exposed by this runtime',
    'Physical cores (host-visible)': psutil.cpu_count(logical=False),
    'Logical processors (host-visible)': os.cpu_count(),
    'RAM GiB (host-visible)': round(psutil.virtual_memory().total / 2**30, 2),
    'HiGHS engine': highspy.Highs().version(),
    'Packages': {name: version(name) for name in ['pyomo', 'highspy', 'networkx', 'numpy', 'pandas']},
    'HiGHS threads': 1,
    'Graph and weight seed': 2022,
    'Warm-up runs per strategy and size': 1,
    'Timed repetitions': 3,
    'Background load / VM resource quotas': 'not controlled or measured',
}
display(pd.Series(environment, name='Current experiment environment'))


In [ ]:
def GenerateGraph(n, seed=2022):
    if n < 4:
        raise ValueError('Use at least four nodes for this graph generator.')
    g = nx.connected_watts_strogatz_graph(n, min(10, n - 1), 0.8, seed=seed)
    rng = np.random.default_rng(seed)
    lengths = {edge: int(value) for edge, value in zip(g.edges(), rng.integers(1, 10, g.number_of_edges()))}
    nx.set_edge_attributes(g, lengths, 'length')
    return g


In [ ]:
strategies= [ SolveAsLO, SimpleDijkstra, Dijkstra ]

In [ ]:
def DoThese(n_max, strategies, step=10, repeats=3):
    if repeats < 1:
        raise ValueError('Use at least one timed repetition.')
    timings = pd.DataFrame(index=range(10, n_max, step),
                           columns=[f.__name__ for f in strategies], dtype=float)
    values = timings.copy()
    samples = []
    for n in timings.index:
        g = GenerateGraph(n)
        reference = nx.shortest_path_length(g, 0, n - 1, weight='length')
        for strategy in strategies:
            _, warmup_value = strategy(g, 0, n - 1)
            if not np.isclose(warmup_value, reference, rtol=1e-9, atol=1e-8):
                raise AssertionError(f'{strategy.__name__} disagrees on weighted distance for n={n}.')
            seconds = []
            for repeat in range(repeats):
                started = pc()
                _, value = strategy(g, 0, n - 1)
                elapsed = pc() - started
                if not np.isclose(value, reference, rtol=1e-9, atol=1e-8):
                    raise AssertionError(f'{strategy.__name__} returned an incorrect weighted distance.')
                seconds.append(elapsed)
                samples.append({'nodes': n, 'strategy': strategy.__name__,
                                'repeat': repeat + 1, 'seconds': elapsed, 'distance': value})
            timings.at[n, strategy.__name__] = np.median(seconds)
            values.at[n, strategy.__name__] = value
    timings.attrs['samples'] = samples
    timings.attrs['environment'] = environment.copy()
    return timings, values


In [ ]:
lo_timings, lo_values = DoThese(100, [SolveAsLO, SimpleDijkstra, Dijkstra])
display(lo_values.rename_axis('Nodes'))
lo_timings.plot(xlabel='Nodes', ylabel='Median wall-clock seconds',
                title='Same weighted problem: model building and solve versus algorithms')
plt.show()


In [ ]:
algorithm_timings, algorithm_values = DoThese(1000, [SimpleDijkstra, Dijkstra], step=100)
display(algorithm_values.rename_axis('Nodes'))
algorithm_timings.plot(xlabel='Nodes', ylabel='Median wall-clock seconds',
                       title='Minimum scan versus heap: three timed repetitions')
plt.show()


## Install a profiler now and inspect the implementations

We install `line_profiler` at this stage, after the ordinary timing experiment.
The profiler measures individual lines and adds instrumentation overhead: do not
mix these times with the earlier unprofiled measurements.

The default profiling graph has 2,000 nodes. Only enable the 100,000-node option
deliberately: the simple minimum-scan implementation can take several minutes
or longer, especially with instrumentation in a shared Colab runtime. Both
profiled functions still receive the very same weighted graph.


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'line_profiler': 'line_profiler'}
ensure_packages(required_packages)


In [ ]:
%load_ext line_profiler

In [ ]:
RUN_LARGE_PROFILE = False
profile_size = 100000 if RUN_LARGE_PROFILE else 2000
g = GenerateGraph(profile_size)
print(f'Profiling on {g.number_of_nodes():,} nodes and {g.number_of_edges():,} edges.')


In [ ]:
%lprun -u 1e-3 -f SimpleDijkstra SimpleDijkstra(g,0,len(g.nodes())-1)

In [ ]:
%lprun -u 1e-3 -f Dijkstra Dijkstra(g,0,len(g.nodes())-1)

## Compare with NetworkX on weighted shortest paths

The edge attribute in this lesson is named `length`, so pass `weight='length'`
explicitly. Omitting it asks NetworkX to minimize the number of edges instead.
One call to `single_source_dijkstra` returns both the route and its weighted
distance; we do not time two searches just to retrieve those two outputs.


In [ ]:
def nx_sp(g, s, t):
    validate_path_input(g, s, t)
    distance, path = nx.single_source_dijkstra(g, s, target=t, weight='length')
    return path, distance


In [ ]:
networkx_timings, networkx_values = DoThese(1000, [Dijkstra, nx_sp], step=100)
display(networkx_values.rename_axis('Nodes'))
networkx_timings.plot(xlabel='Nodes', ylabel='Median wall-clock seconds',
                      title='Heap implementations solving the same weighted problem')
plt.show()


## Explain the experiment

- Why can omitting a weight argument change the optimization problem?
- Where does the minimum-scan implementation spend its time, and how does a heap change that?
- Why does the LP timing include more than the optimizer's search time?
- Can two different routes both be correct? What must still agree?
- Change the seed and rerun a comparison. Do the answers still agree, and how stable are the timings?

The experiment compares implementations and their measured costs in one runtime.
It does not establish a universal speedup or replace complexity analysis.
